<a href="https://colab.research.google.com/github/yUHenGZhOng/School-Project/blob/main/%E6%83%B3%E5%86%99%E6%B5%81%E5%A4%84%E7%90%86%E7%89%88%E6%9C%AC%E7%9A%84%E5%8D%8A%E6%88%90%E5%93%81.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/serengil/deepface.git

  Cloning https://github.com/serengil/deepface.git to /tmp/pip-req-build-hquly0t6
  Running command git clone --filter=blob:none --quiet https://github.com/serengil/deepface.git /tmp/pip-req-build-hquly0t6
  Resolved https://github.com/serengil/deepface.git to commit 4ac3f07510a50f608a7027d5c1b137fb4f35e7d3
  Preparing metadata (setup.py) ... done


In [ ]:
import deepface
print(f"DeepFace 版本: {deepface.__version__}")

try:
    from deepface.commons import distance as dst
    print("从 deepface.commons 导入 distance 成功！")
    print("可以正常使用最终版的代码了。")
except ImportError as e:
    print(f"导入失败: {e}")
    print("如果看到此消息，说明问题依然存在，可能需要检查库的依赖冲突。")

DeepFace 版本: 0.0.94
导入失败: cannot import name 'distance' from 'deepface.commons' (/usr/local/lib/python3.11/dist-packages/deepface/commons/__init__.py)
如果看到此消息，说明问题依然存在，可能需要检查库的依赖冲突。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2
import pandas as pd
from deepface import DeepFace
import os
import numpy as np


In [ ]:
!pip install --upgrade deepface

In [ ]:
# from deepface.commons.distance import findCosineDistance, findEuclideanDistance

In [ ]:
def analyze_video_test_verify_with_visuals(video_path, interval_seconds=1, output_dir='debug_output'):
    """
    测试版本：使用Greedy Matching算法，但比对核心使用DeepFace.verify()和FaceNet512模型。
    新增了强大的可视化验证功能，用于保存图片以供人工核查。
    """
    if not os.path.exists(video_path):
        print(f"错误：视频文件未找到 -> {video_path}")
        return None

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"错误：无法打开视频文件 {video_path}")
        return None

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_increment = int(fps * interval_seconds) if fps > 0 else 30
    if frame_increment < 1: frame_increment = 1

    # 创建用于保存调试图片的总输出目录
    os.makedirs(output_dir, exist_ok=True)
    full_frames_dir = os.path.join(output_dir, 'full_frames')
    cropped_faces_dir = os.path.join(output_dir, 'cropped_faces')
    os.makedirs(full_frames_dir, exist_ok=True)
    os.makedirs(cropped_faces_dir, exist_ok=True)

    print(f"视频信息: FPS ≈ {fps:.2f}, 每隔 {interval_seconds} 秒 (≈{frame_increment} 帧) 分析一次。")
    print(f"调试图片将保存到: {output_dir} 文件夹")

    known_faces_db = {}
    person_id_counter = 0
    MODEL_NAME = 'FaceNet512'

    all_results = []
    frame_number = 0

    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()
        if not ret: break

        print(f"--- 正在处理第 {frame_number} 帧 (模型: {MODEL_NAME}, 核心: verify) ---")

        try:
            # 人脸检测与特征提取
            faces_in_frame = DeepFace.analyze(img_path=frame, actions=['emotion'], enforce_detection=False, detector_backend='opencv')

            possible_matches = []
            face_embeddings_cache = {} # 缓存新的人脸id向量
            detected_faces_cache = {} # 缓存人脸图像

            for idx, face_data in enumerate(faces_in_frame):
                x, y, w, h = face_data['region']['x'], face_data['region']['y'], face_data['region']['w'], face_data['region']['h']
                detected_face_img = frame[y:y+h, x:x+w].copy()
                detected_faces_cache[idx] = detected_face_img

                # 使用 verify 进行比对，但是veryfy函数的embedding对比信息似乎调用失败？
                for person_id, person_repr_embedding in known_faces_db.items():
                    try:
                        verification = DeepFace.verify(
                            img1_path=detected_face_img,
                            img2_path=person_repr_embedding,
                            model_name=MODEL_NAME,
                            enforce_detection=False
                        )
                        if verification['verified']:
                            possible_matches.append((idx, person_id, verification['distance']))
                    except Exception as e:
                        print(f"Verify比对出错: {e}")

            # 贪心匹配算法
            possible_matches.sort(key=lambda x: x[2])
            assignments = {}
            assigned_ids = set()
            for face_idx, person_id, distance in possible_matches:
                if face_idx not in assignments and person_id not in assigned_ids:
                    assignments[face_idx] = person_id
                    assigned_ids.add(person_id)

            # 分配ID、保存结果和调试图片
            frame_processed = False
            for idx, face_data in enumerate(faces_in_frame):
                assigned_id = None
                if idx in assignments:
                    assigned_id = assignments[idx]
                else:
                    try:
                        embedding = DeepFace.represent(img_path=detected_faces_cache[idx], model_name=MODEL_NAME, enforce_detection=False)[0]["embedding"]
                        person_id_counter += 1
                        assigned_id = f'person_{person_id_counter}'
                        known_faces_db[assigned_id] = embedding
                        print(f"检测到新成员！分配ID: {assigned_id}")
                    except:
                        continue

                if assigned_id:
                    frame_processed = True
                    # 【新功能】保存带ID的人脸切片图
                    person_dir = os.path.join(cropped_faces_dir, assigned_id)
                    os.makedirs(person_dir, exist_ok=True)
                    face_filename = os.path.join(person_dir, f"frame_{frame_number}.jpg")
                    cv2.imwrite(face_filename, detected_faces_cache[idx])

                    # 在完整帧上绘制矩形框和ID
                    x, y, w, h = face_data['region']['x'], face_data['region']['y'], face_data['region']['w'], face_data['region']['h']
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    cv2.putText(frame, assigned_id, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

                    # 记录分析结果
                    result = {'frame': frame_number, 'person_id': assigned_id, **face_data}
                    all_results.append(result)

            # 如果该帧被处理过，则保存带有标注的完整帧图像
            if frame_processed:
                full_frame_filename = os.path.join(full_frames_dir, f"frame_{frame_number}.jpg")
                cv2.imwrite(full_frame_filename, frame)
                print(f"本帧分析完成，已保存标注图片。 IDs: {[r['person_id'] for r in all_results if r['frame'] == frame_number]}")

        except Exception as e:
            print(f"处理第 {frame_number} 帧时发生主循环错误: {e}")

        frame_number += frame_increment

    cap.release()
    print("\n--- [测试] 视频处理完成！---")
    return pd.DataFrame(all_results) if all_results else pd.DataFrame()

In [ ]:
# 主程序
if __name__ == '__main__':
    VIDEO_FILE_PATH = '/content/drive/MyDrive/FacialExpAnalysis/3.mp4'  # 源视频流路径指定
    ANALYSIS_INTERVAL_SECONDS = 60
    OUTPUT_DEBUG_DIR = '/content/drive/MyDrive/FacialExpAnalysis/deepface/testrun_output/'

    df = analyze_video_test_verify_with_visuals(
        video_path=VIDEO_FILE_PATH,
        interval_seconds=ANALYSIS_INTERVAL_SECONDS,
        output_dir=OUTPUT_DEBUG_DIR
    )

    if df is not None and not df.empty:
        OUTPUT_CSV_PATH = f"/content/drive/MyDrive/FacialExpAnalysis/deepface/results/{file_name_without_ext}_emotions_interval_{ANALYSIS_INTERVAL_SECONDS}s.csv"
        cols_order = [
            'frame', 'timestamp_seconds', 'person_id', 'dominant_emotion',
            'happy', 'sad', 'angry', 'surprise', 'fear', 'disgust', 'neutral'
        ]
        df_filtered = df[[col for col in cols_order if col in df.columns]]
        df_filtered.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
        print(f"\n分析结果已成功保存到 -> {OUTPUT_CSV_PATH}")
        print("\n数据预览:")
        print(df_filtered.head())

视频信息: FPS ≈ 25.00, 每隔 60 秒 (≈1500 帧) 分析一次。
调试图片将保存到: /content/drive/MyDrive/FacialExpAnalysis/deepface/testrun_output/ 文件夹
--- 正在处理第 0 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 1500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 3000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 4500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 6000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 7500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 9000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 10500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 12000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 13500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 15000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 16500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 18000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 19500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 21000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 22500 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 24000 帧 (模型: FaceNet512, 核心: verify) ---
--- 正在处理第 25500 帧 (模